<a href="https://colab.research.google.com/github/JoshuaNiel/CS452/blob/main/mongo/Step1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Let's learn about Mongo!
*Learning Goal*: Using a sandbox environment create, update, and delete documents. Query documents that other students are creating at the same time. Intention is to go through this together as a class and explain syntax step by step.

In [1]:
# Install the pymongo library
%pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.4 MB/s eta 0:00:00


In [2]:
# Connect to a provided sandbox environment

import pymongo

user = "class"
password = "184vLpDKvOhvv528"
cluster = "cluster0"
dnsprefix = "wvdjn"
connectionUrl = f"mongodb+srv://{user}:{password}@{cluster}.{dnsprefix}.mongodb.net/"
client = pymongo.MongoClient(connectionUrl)
print(f"Ping result: {client.admin.command('ping')}")

db = client.get_database("sandbox")

# accessing db.students creates or accesses the collection "students" within
# the "sandbox" db
students = db.students



Ping result: {'ok': 1}


In [4]:
# Put your information in these variables and insert yourself into students!

netId = input("Enter netId (this just needs to be unique): ")
name = input("Your name: ")
favorite_color = input("Favorite color: ")
number = int(input("Number between 0 and 9 inclusive: "))


# Creates a dictionary object in python with your data
number = number % 10
me = {
    "_id": netId,
    "name": name,
    "color": favorite_color,
    "num": number
}

students.insert_one(me)

Enter netId (this just needs to be unique): jrn76
Your name: Joshua
Favorite color: purple!
Number between 0 and 9 inclusive: 0


InsertOneResult('jrn76', acknowledged=True)

In [5]:
# Query to see yourself in the collection!
result = students.find({"_id": netId})
list(result)

[{'_id': 'jrn76', 'name': 'Joshua', 'color': 'purple!', 'num': 0}]

In [6]:
# Are there students who like your same color (limit to 10)
result = students.find({"color": favorite_color}).limit(10)
list(result)

[{'_id': 'jrn76', 'name': 'Joshua', 'color': 'purple!', 'num': 0}]

In [7]:
# Are there students who picked your same number? (limit to 10)
result = students.find({"num": number}).limit(10)
list(result)

[{'_id': 'ianjr',
  'name': 'Ian Robertson',
  'color': 'Green',
  'num': 0,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Do Homework'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Just be vibin'},
   {'order': 4, 'name': 'Family Home Evening'}]},
 {'_id': 'jrn76', 'name': 'Joshua', 'color': 'purple!', 'num': 0}]

In [8]:
# How about we make that a little more easy to read!

# use the second parameter of find which is "project" to limit the fields
result = students.find(
        {"num": number},  # match or find clause
        {"_id":0, "name":1}  # project statement for changing the output
    ).limit(10)
list(result)

[{'name': 'Ian Robertson'}, {'name': 'Joshua'}]

In [9]:
# Let's update your record with more information!
achievement = "I can run an update!"
students.update_one({"_id": netId}, {"$set": {"achievement": achievement}})


UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff00000000000000b0'), 'opTime': {'ts': Timestamp(1775054953, 23), 't': 176}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1775054953, 23), 'signature': {'hash': b'\\\xd6\xfcK\x93\xab9&e\x1eq\x7fSl\x91&\xd2\xe3\x80#', 'keyId': 7592293571235938313}}, 'operationTime': Timestamp(1775054953, 23), 'updatedExisting': True}, acknowledged=True)

In [10]:
# Let's see if the update worked!

result = students.find({"_id": netId})
list(result)

[{'_id': 'jrn76',
  'name': 'Joshua',
  'color': 'purple!',
  'num': 0,
  'achievement': 'I can run an update!'}]

In [11]:
# Let's see if anyone with similar color preference or number
# has also been able to update their record.

result = students.find({"$or": [{"color": favorite_color}, {"num": number}]})
list(result)


[{'_id': 'ianjr',
  'name': 'Ian Robertson',
  'color': 'Green',
  'num': 0,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Do Homework'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Just be vibin'},
   {'order': 4, 'name': 'Family Home Evening'}]},
 {'_id': 'jrn76',
  'name': 'Joshua',
  'color': 'purple!',
  'num': 0,
  'achievement': 'I can run an update!'}]

In [12]:
# Let's see who prefers bigger numbers

result = students.find({"num": {"$gt": 5}}, {"_id":0, "name": 1, "number": 1})
list(result)

[{'name': 'Rachel'},
 {'name': 'Merica Rowley'},
 {'name': 'Melvin Whitaker'},
 {'name': 'Michael Reynolds'},
 {'name': 'Jose C'},
 {'name': 'Cam'},
 {'name': 'Patrick'}]

In [13]:
# Let's see who prefers smaller numbers

result = students.find({"num": {"$lt": 5}}, {"_id":0, "name": 1, "number": 1})
list(result)

[{'name': 'Ethan Dye'},
 {'name': 'Ian Robertson'},
 {'name': 'Kobey'},
 {'name': 'Caleb Calderwood'},
 {'name': 'Tretn'},
 {'name': 'Joshua'}]

In [14]:
# Let's see who prefers numbers between 3 and 7 (inclusive)
result = students.find(
        {
            "$and": [  # all conditions in this list need to be true
                {"num": {"$gte": 3}}, # greater than or equal to 3
                {"num": {"$lte": 7}}  # less than or equal to 7
            ]
        },
        {"_id":0, "name": 1, "number": 1} # project out just name and number
    )
list(result)

[{'name': 'Ethan Dye'},
 {'name': 'Merica Rowley'},
 {'name': 'Melvin Whitaker'},
 {'name': 'Kobey'},
 {'name': 'Michael Reynolds'},
 {'name': 'Caleb Calderwood'},
 {'name': 'Jose C'},
 {'name': 'James Chan'},
 {'name': 'Cam'},
 {'name': 'john'}]

In [15]:
# What are all the distinct numbers and colors that people picked?
print(f"Numbers: {students.distinct('num')}")
print(f"Colors: {students.distinct('color')}")

Numbers: [0, 1, 3, 4, 5, 7, 8]
Colors: ['5', 'Blue', 'Green', 'Pink', 'Purple', 'Yellow', 'green', 'ok', 'purple!']


In [16]:
# Let's get serious! What goals do you have for the rest of the day?
# Update your record with those goals! <----<<

my_goals = [
    {"order": 1, "name": "Eat dinner"},
    {"order": 2, "name": "Say my prayers"},
    {"order": 3, "name": "Sleep"},
    # update these goals maybe add order #4? (keep the schema) <---<<
    {"order": 4, "name": "Finish all assignments"},
    {"order": 5, "name": "Send out many job applications"}
]

students.update_one(
    {"_id": netId},
    {
        "$set": {
            "goals": my_goals
        }
    }
)


UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff00000000000000b0'), 'opTime': {'ts': Timestamp(1775055375, 10), 't': 176}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1775055375, 10), 'signature': {'hash': b'\xc6\x90\x13w\xc9d3M\xfa\xeb\x1c\x1b\xd7\xbc\x1d\xd3\xcaa[@', 'keyId': 7592293571235938313}}, 'operationTime': Timestamp(1775055375, 10), 'updatedExisting': True}, acknowledged=True)

In [17]:
# Who's next goal "order 1" is something other than eating dinner?

result = students.find({
        "goals.order": 1,
        "goals.name": {"$ne": "Eat dinner"}
    }).limit(10)

list(result)

[{'_id': 'mrtops',
  'name': 'Ethan Dye',
  'color': 'Green',
  'num': 3,
  'goals': [{'order': 1, 'name': 'Homework'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Sleep'}]},
 {'_id': 'mericar',
  'name': 'Merica Rowley',
  'color': 'Blue',
  'num': 7,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Organize FHE'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Help my dad with his website'},
   {'order': 4, 'name': 'Get Mongo homework done'}]},
 {'_id': 'ianjr',
  'name': 'Ian Robertson',
  'color': 'Green',
  'num': 0,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Do Homework'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Just be vibin'},
   {'order': 4, 'name': 'Family Home Evening'}]},
 {'_id': 'mw962',
  'name': 'Melvin Whitaker',
  'color': 'Purple',
  'num': 7,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Finish homework'},
  

In [18]:
# Who's next goal "order 1" is eating dinner?

result = students.find({
        "goals.order": 1,
        "goals.name": "Eat dinner"
    }).limit(10)

list(result)

[{'_id': 'rwhite37',
  'name': 'Rachel',
  'color': 'Pink',
  'num': 8,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Eat dinner'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Sleep'}]},
 {'_id': 'kobeynw',
  'name': 'Kobey',
  'color': 'Green',
  'num': 4,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Eat dinner'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Sleep'}]},
 {'_id': 'jmcr94',
  'name': 'Jose C',
  'color': 'Green',
  'num': 7,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Eat dinner'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Sleep'}]},
 {'_id': 'jamcha99',
  'name': 'James Chan',
  'color': '5',
  'num': 5,
  'achievement': 'I can run an update!',
  'goals': [{'order': 1, 'name': 'Eat dinner'},
   {'order': 2, 'name': 'Say my prayers'},
   {'order': 3, 'name': 'Sleep'}]},
 {'_id': 'jsbarre12',
  'name': 'john',
 

In [19]:
# Who has an order 4 goal and what is it? Make it look nice by just showing the requested information.

result = students.find(
    {"goals": {"$elemMatch": {"order": 4}}},
    {"name": 1, "goals.$": 1, "_id": 0}
)
list(result)

[{'name': 'Merica Rowley',
  'goals': [{'order': 4, 'name': 'Get Mongo homework done'}]},
 {'name': 'Ian Robertson',
  'goals': [{'order': 4, 'name': 'Family Home Evening'}]},
 {'name': 'Michael Reynolds',
  'goals': [{'order': 4, 'name': 'Do leetcode problem for 393'}]},
 {'name': 'Caleb Calderwood', 'goals': [{'order': 4, 'name': 'Cry'}]},
 {'name': 'Joshua', 'goals': [{'order': 4, 'name': 'Finish all assignments'}]}]

### Explanation on the last code courtesy of ChatGPT (and formatted in markdown by GPT)

**Explanation:**

- `{"goals": {"$elemMatch": {"order": 4}}}` specifies the condition where there is at least one element in the `goals` array with order equal to 4.
- `{"name": 1, "goals.$": 1, "_id": 0}` specifies the projection to include only the `name` field and the matching goal with order 4 (`goals.$` represents the matched element from the `goals` array), while excluding the `_id` field.



In [ ]:
# If you'd like to preserve your record for students who were sick and
# couldn't attend class then you are done!

# If you'd like to remove your record from this dataset then uncomment the
# following and remove it!

#students.delete_one({"_id": netId})

Maybe you could visit the docs: https://www.mongodb.com/docs/manual/

Or maybe go give chatgpt your schema and some sample code and ask it to help you write some interesting queries!

In [20]:
# Aggregation - the average number that people picked

result = students.aggregate(
    [
        {
            "$group":{
                "_id": None,
                "nums": {"$avg": "$num"}
            }
        }
    ]
)

list(result)

[{'_id': None, 'nums': 4.866666666666666}]